In [24]:
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_information_units(sentence: str):
    doc = nlp(sentence)
    units = []
    # Named Entities, Pronouns, Noun Phrases, and Verb Phrases
    for ent in doc.ents:
        units.append(ent.text)
    for chunk in doc.noun_chunks:
        units.append(chunk.text)
    for token in doc:
        if token.pos_ in ["PRON", "VERB"]:
            units.append(token.text)
    return list(set(units)) # Deduplicate

## Information Unit Extraction
Extract named entities, pronouns, nouns, and verb phrases from sentences

In [25]:
import sys
import os
import json
import logging
from typing import List, Dict, Tuple
import pandas as pd
from datetime import datetime

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("DecontextTest")

print("✓ Imports successful")

✓ Imports successful


In [26]:
!pip install transformers torch spacy rank-bm25
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 14.0 MB/s  0:00:01m0:00:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## Setup and Imports

# Decontextualization Testing
Tests the QA-based decontextualization method from the paper:

## Pipeline Steps:
1. **Question Generation**: Extract ambiguous information units (entities, pronouns, nouns) and generate questions
2. **Question Answering**: Answer questions using the full document context via BM25 + QA
3. **QA-to-Context**: Convert QA pairs to declarative sentences using BART/T5
4. **Sentence Rewriting**: Use T5 to rewrite sentences with generated context

Based on: "Modeling claim extraction with QA-based decontextualization"

In [27]:
import torch
import time
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

class Decontextualizer:
    def __init__(self, use_gpu: bool = True, max_workers: int = 4):
        self.device = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"
        self.max_workers = max_workers
        print(f"Initializing Decontextualizer on {self.device}...")

        # 1. Question Generation (QG) - Remains the same
        self.qg_model_name = "mrm8488/t5-base-finetuned-question-generation-ap"
        self.qg_tokenizer = AutoTokenizer.from_pretrained(self.qg_model_name)
        self.qg_model = AutoModelForSeq2SeqLM.from_pretrained(self.qg_model_name).to(self.device)

        # 2. Question Answering (QA) - Remains the same
        self.qa_pipe = pipeline(
            "question-answering", 
            model="deepset/roberta-base-squad2",
            device=0 if self.device == "cuda" else -1
        )

        # 3. Use 't5-base' instead of 't5-v1_1-base'
        # t5-base is fine-tuned and understands instructions.
        self.gen_model_name = "t5-base" 
        self.gen_tokenizer = AutoTokenizer.from_pretrained(self.gen_model_name)
        self.gen_model = AutoModelForSeq2SeqLM.from_pretrained(self.gen_model_name).to(self.device)

    def _generate(self, prompt: str, tokenizer, model, max_length: int = 128):
        # Add truncation=True to prevent the 512 token error
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(self.device)
        
        # Add repetition_penalty and no_repeat_ngram_size to stop the 'question: question:' loops
        outputs = model.generate(
            **inputs, 
            max_length=max_length, 
            num_beams=4, 
            repetition_penalty=2.5,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    def _process_unit(self, unit: str, sentence: str, article_paragraphs: list) -> str:
        try:
            q_prompt = f"answer: {unit} context: {sentence}"
            question = self._generate(q_prompt, self.qg_tokenizer, self.qg_model)

            relevant_evidence = [p for p in article_paragraphs if unit.lower() in p.lower()]
            evidence = relevant_evidence[0] if relevant_evidence else " ".join(article_paragraphs[:3])

            qa_res = self.qa_pipe(question=question, context=evidence)
            
            if qa_res['score'] > 0.35:
                # Use a more standard T5 prompt format
                qa2d_prompt = f"make a sentence: question: {question} answer: {qa_res['answer']}"
                return self._generate(qa2d_prompt, self.gen_tokenizer, self.gen_model)
        except:
            return None
        return None

    def decontextualize(self, sentence: str, full_document: str, units: list) -> dict:
        start_time = time.time()
        paragraphs = full_document.split('\n\n')
        
        # Limit units to reduce processing time
        filtered_units = [u for u in units if len(u.split()) < 5][:4] 

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = [executor.submit(self._process_unit, u, sentence, paragraphs) for u in filtered_units]
            context_sentences = [f.result() for f in futures if f.result() is not None]

        full_context = " ".join(context_sentences)
        
        # Using 'elaborate:' or 'summarize:' as T5 recognizes these tasks better
        final_prompt = f"summarize: {full_context} {sentence}"
        result = self._generate(final_prompt, self.gen_tokenizer, self.gen_model, max_length=256)

        return {
            "original": sentence,
            "context_gathered": full_context,
            "decontextualized": result,
            "runtime": time.time() - start_time,
            "units_processed": len(filtered_units)
        }

## Load Test Article

## Decontextualizer Class
Implements the full QA-based decontextualization pipeline

In [28]:
# Load test article
json_path = 'article2.json'

with open(json_path, 'r') as f:
    data = json.load(f)

article_title = data.get('article_title', 'Unknown Title')
article_text = data.get('article_text', '')
article_url = data.get('article_url', '')

print(f"Article: {article_title}")
print(f"Text length: {len(article_text)} characters")
print(f"\nFirst 300 characters:")
print(article_text[:300] + "...")

Article: What next for Venezuela? What leaders and experts said at Davos
Text length: 9670 characters

First 300 characters:
GEOGRAPHIES IN DEPTH
What next for Venezuela? What leaders and experts said at Davos
Jan 23, 2026

Ngaire Woods: The international community has to 'create the conditions for national consensus to take place.' Image: World Economic Forum

Pablo Uchoa
Writer, Forum Stories
This article is part of:
Wo...


## Extract Test Sentences
Get a sample of sentences that likely need decontextualization (contain pronouns, ambiguous references)

In [29]:
import spacy
import re

# Load spacy
nlp = spacy.load("en_core_web_sm")

# Clean and split into sentences
text = re.sub(r'\s+', ' ', article_text)
doc = nlp(text)
all_sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.split()) > 5]

# Find sentences that likely need decontextualization
# (contain pronouns like he, she, it, they, this, that)
ambiguous_sentences = []
pronoun_patterns = ['he', 'she', 'it', 'they', 'this', 'that', 'these', 'those', 'his', 'her', 'their']

for sent in all_sentences[:30]:  # Check first 30 sentences
    sent_lower = sent.lower()
    if any(f' {pron} ' in f' {sent_lower} ' for pron in pronoun_patterns):
        ambiguous_sentences.append(sent)

print(f"Total sentences: {len(all_sentences)}")
print(f"Sentences with potential ambiguity: {len(ambiguous_sentences)}")
print("\nAmbiguous sentences (first 10):")
for i, sent in enumerate(ambiguous_sentences[:10], 1):
    print(f"\n{i}. {sent}")

Total sentences: 58
Sentences with potential ambiguity: 20

Ambiguous sentences (first 10):

1. Image: World Economic Forum Pablo Uchoa Writer, Forum Stories This article is part of: World Economic Forum Annual Meeting Venezuela faces profound political and economic uncertainty in the wake of the US military intervention that captured Nicolás Maduro on 3 January.

2. Venezuela’s political future and economic recovery have been debated across Davos this week, from Latin American leaders to geopolitical and energy experts.

3. Venezuela is facing huge uncertainties — both political and economic — following the US military intervention that ousted and captured Nicolás Maduro on 3 January.

4. And in Davos this week, the country's future was part of the conversations among Latin American leaders and geopolitical and energy experts across several panels during the Annual Meeting.

5. In a session titled Venezuela: What Next?, a group of experts converged on a sobering assessment that Washin

## Initialize Decontextualizer
Load all required models (this may take a few minutes)

In [30]:
print("Initializing Decontextualizer...")
print("This will load 4 models: QG, QA, QA2D, and T5 Rewriter")
print("Please wait...\n")

try:
    decontextualizer = Decontextualizer()
    print("✓ Decontextualizer initialized successfully!")
except Exception as e:
    print(f"✗ Error initializing: {e}")
    raise

Initializing Decontextualizer...
This will load 4 models: QG, QA, QA2D, and T5 Rewriter
Please wait...

Initializing Decontextualizer on cuda...


httpx - INFO - HTTP Request: HEAD https://huggingface.co/mrm8488/t5-base-finetuned-question-generation-ap/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mrm8488/t5-base-finetuned-question-generation-ap/c81cbaf0ec96cc3623719e3d8b0f238da5456ca8/config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/mrm8488/t5-base-finetuned-question-generation-ap/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mrm8488/t5-base-finetuned-question-generation-ap/c81cbaf0ec96cc3623719e3d8b0f238da5456ca8/tokenizer_config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/mrm8488/t5-base-finetuned-question-generation-ap/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
httpx - INFO - HTTP Request: GET https:/

✓ Decontextualizer initialized successfully!


## Test Decontextualization Pipeline
Process sample sentences and compare original vs decontextualized versions

In [32]:
import time

# Select test sentences
test_sentences = ambiguous_sentences[5:9]
final_results = []

print("=" * 100)
print(f"{'DECONTEXTUALIZATION REPORT':^100}")
print("=" * 100)

for idx, sentence in enumerate(test_sentences, 1):
    print(f"\n[CLAIM {idx}/{len(test_sentences)}]")
    print(f"Original: {sentence}")
    
    try:
        # Pre-extract units for the class (assuming extract_information_units is defined)
        units = extract_information_units(sentence)
        
        # Call the master method within the class
        # This handles the parallel QG -> QA -> QA2D loop and final rewrite
        res = decontextualizer.decontextualize(
            sentence=sentence, 
            full_document=article_text, 
            units=units
        )
        
        print(f"Context Found: {res['context_gathered'][:120]}...")
        print(f"Rewritten:    {res['decontextualized']}")
        print(f"Efficiency:   {res['runtime']:.2f}s (via {res['units_processed']} info units)")
        
        final_results.append(res)
        
    except Exception as e:
        print(f"✗ Execution Error: {e}")
        final_results.append({
            "original": sentence, 
            "decontextualized": f"FAILED: {str(e)}", 
            "runtime": 0,
            "success": False
        })

print("\n" + "=" * 100)
print(f"{'PIPELINE COMPLETE':^100}")
print("=" * 100)

                                     DECONTEXTUALIZATION REPORT                                     

[CLAIM 1/4]
Original: Ricardo Hausmann, Founder and Director of Harvard University’s Growth Lab — and a former Venezuelan Minister of Planning — argued that what is being framed as "stability" today is, in fact, repression: the continued detention of political prisoners and severe curtailment on basic freedoms.
Context Found: False False False...
Rewritten:    former Venezuelan minister of planning argued that what is being framed as "stability" today is, in fact, repression .
Efficiency:   12.06s (via 4 info units)

[CLAIM 2/4]
Original: In his view, a climate of coercion will deter Venezuela's roughly eight-million-strong diaspora from returning to help drive the country’s economic reconstruction.
Context Found: entailment...
Rewritten:    climate of coercion will deter Venezuela's roughly eight-million-strong diaspora from returning to help drive country’s economic reconstruction .
